# 🧪 Mini‑Lakehouse Project (Retail) — Bronze → Silver → Gold
**Entrega sugerida:** 2025-08-09 + 7 días (ajústalo en el enunciado).  
**Entorno:** Databricks Free Edition (hive_metastore, tablas *managed* Delta).

**Objetivo:** Construir un pipeline batch (con opción de simular incremental) usando el dataset de **Retail** incluido en Databricks:  
`dbfs:/databricks-datasets/retail-data/by-day/*.csv`

**Instrucciones generales**
- Trabaja en un **solo notebook** (este).
- Crea un **schema** dedicado para tus tablas (ej.: `bd_retail_tunombre`).
- Usa **Delta** en todas las capas.
- Implementa: limpieza, *casts*, derivaciones, deduplicación (window), agregaciones (KPIs), *views*, y operaciones Delta (history, time travel, OPTIMIZE/ZORDER, VACUUM).
- Marca los **TODO** y completa donde se indica.


## 0) Parámetros y Setup

In [0]:

# TODO: Reemplaza con tu identificador corto (solo letras/números/guiones bajos)
student = "TU_NOMBRE"

# Construimos el nombre del schema (base de datos) para tus tablas managed
import re
student_slug = re.sub(r"[^a-zA-Z0-9_]", "_", student).lower()
db = f"bd_retail_{student_slug}"

print("Schema destino:", db)

# Creamos el schema si no existe y lo seleccionamos
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

# Opcional: ver versión de Spark/DBR
spark.sparkContext.version


## 1) Bronze — Ingesta cruda con esquema explícito
- Lee CSVs desde `dbfs:/databricks-datasets/retail-data/by-day/*.csv`.
- Usa **esquema explícito** (lee todo como *string* y tipifica en Silver).
- Escribe una **tabla Delta managed** `retail_bronze`.


In [0]:

from pyspark.sql.types import *
from pyspark.sql.functions import *

raw_schema = StructType([
    StructField("InvoiceNo",    StringType(), True),
    StructField("StockCode",    StringType(), True),
    StructField("Description",  StringType(), True),
    StructField("Quantity",     StringType(), True),   # tipificaremos en Silver
    StructField("InvoiceDate",  StringType(), True),   # tipificaremos en Silver (timestamp)
    StructField("UnitPrice",    StringType(), True),   # tipificaremos en Silver
    StructField("CustomerID",   StringType(), True),   # tipificaremos en Silver (int)
    StructField("Country",      StringType(), True)
])

raw_path = "dbfs:/databricks-datasets/retail-data/by-day/*.csv"

bronze_df = (spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv(raw_path)
)

# TODO: Inspecciona datos crudos
display(bronze_df.limit(10))

# Persistimos como tabla Delta managed
(bronze_df.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(f"{db}.retail_bronze")
)

spark.sql(f"DESCRIBE HISTORY {db}.retail_bronze").show(truncate=False)
spark.table(f"{db}.retail_bronze").count()


## 2) Silver — Limpieza, tipificación y deduplicación
**Transformaciones mínimas:**
- `trim/lower` en campos de texto clave
- Conversiones: `Quantity -> int`, `UnitPrice -> double`, `CustomerID -> int`, `InvoiceDate -> timestamp`
- Derivadas: `sale_timestamp`, `sale_date`, `line_total = Quantity * UnitPrice`, `is_return = Quantity < 0`
- **Reglas de calidad**: descarta filas inválidas (nulos en claves, precios/cantidades no numéricos, etc.)
- **Deduplicación** usando window: particiona por `(InvoiceNo, StockCode)` y conserva la fila más reciente por `sale_timestamp`


In [0]:

from pyspark.sql.window import Window

bronze = spark.table(f"{db}.retail_bronze")

silver_pre = (bronze
    # Normalización básica de texto
    .withColumn("InvoiceNo", trim(col("InvoiceNo")))
    .withColumn("StockCode", trim(lower(col("StockCode"))))
    .withColumn("Description", regexp_replace(trim(lower(col("Description"))), r"\s+", " "))
    .withColumn("Country", trim(initcap(col("Country"))))
    # Tipificación
    .withColumn("Quantity_int",  col("Quantity").cast("int"))
    .withColumn("UnitPrice_dbl", col("UnitPrice").cast("double"))
    .withColumn("CustomerID_int", col("CustomerID").cast("int"))
    .withColumn("sale_timestamp",
                coalesce(
                    to_timestamp(col("InvoiceDate")),                         # intenta parseo por defecto
                    to_timestamp(col("InvoiceDate"), "MM/dd/yyyy HH:mm"),     # formato alternativo 1
                    to_timestamp(col("InvoiceDate"), "dd/MM/yyyy HH:mm")      # formato alternativo 2
                ))
    .withColumn("sale_date", to_date(col("sale_timestamp")))
    .withColumn("line_total", col("Quantity_int") * col("UnitPrice_dbl"))
    .withColumn("is_return", col("Quantity_int") < 0)
)

# Reglas de calidad (ajusta si lo requieres)
silver_clean = (silver_pre
    .filter(col("InvoiceNo").isNotNull())
    .filter(col("StockCode").isNotNull())
    .filter(col("Description").isNotNull())
    .filter(col("sale_timestamp").isNotNull())
    .filter(col("Quantity_int").isNotNull())
    .filter(col("UnitPrice_dbl").isNotNull())
    .filter(col("UnitPrice_dbl") >= 0)
)

# Deduplicación: última observación por (InvoiceNo, StockCode)
w = Window.partitionBy("InvoiceNo","StockCode").orderBy(col("sale_timestamp").desc_nulls_last())
silver_dedup = silver_clean.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

# TODO: Considera mover nulos de CustomerID_int a una "zona de cuarentena" si deseas (opcional)

(silver_dedup
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(f"{db}.retail_silver")
)

display(silver_dedup.limit(10))
spark.sql(f"DESCRIBE HISTORY {db}.retail_silver").show(truncate=False)


## 3) Gold — Métricas y tablas de servicio
Crea al menos tres salidas Gold. Aquí tienes tres ejemplos.


### 3.1) gold_daily_country — ventas diarias por país

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE {db}.gold_daily_country AS
SELECT
  sale_date,
  Country,
  SUM(CASE WHEN is_return = false THEN line_total ELSE 0 END)           AS gross_revenue,
  SUM(CASE WHEN is_return = true  THEN line_total ELSE 0 END)            AS returns_value,
  (SUM(CASE WHEN is_return = false THEN line_total ELSE 0 END) +
   SUM(CASE WHEN is_return = true  THEN line_total ELSE 0 END))          AS net_revenue,
  COUNT(DISTINCT InvoiceNo)                                              AS n_invoices,
  SUM(CASE WHEN is_return = false THEN Quantity_int ELSE 0 END)          AS n_items,
  (CASE WHEN COUNT(DISTINCT InvoiceNo) > 0
        THEN (SUM(CASE WHEN is_return = false THEN line_total ELSE 0 END) +
              SUM(CASE WHEN is_return = true  THEN line_total ELSE 0 END))
             / COUNT(DISTINCT InvoiceNo)
        ELSE NULL END)                                                   AS avg_basket_value
FROM {db}.retail_silver
GROUP BY sale_date, Country
""")
)

display(spark.table(f"{db}.gold_daily_country").orderBy("sale_date","Country").limit(20))


### 3.2) gold_customer_rfm — métricas RFM por cliente

In [0]:

# Determinamos la fecha de referencia (última fecha en Silver)
max_dt = spark.table(f"{db}.retail_silver").selectExpr("max(sale_date) as md").collect()[0]["md"]
print("Max sale_date:", max_dt)

spark.sql(f"""
CREATE OR REPLACE TABLE {db}.gold_customer_rfm AS
WITH base AS (
  SELECT
    CustomerID_int AS customer_id,
    MIN(sale_date) AS first_purchase_date,
    MAX(sale_date) AS last_purchase_date,
    COUNT(DISTINCT InvoiceNo) AS frequency,
    SUM(CASE WHEN is_return = false THEN line_total ELSE 0 END) AS monetary
  FROM {db}.retail_silver
  WHERE CustomerID_int IS NOT NULL
  GROUP BY CustomerID_int
),
scored AS (
  SELECT
    customer_id,
    first_purchase_date,
    last_purchase_date,
    DATEDIFF(date('{max_dt}'), last_purchase_date) AS recency_days,
    frequency,
    monetary,
    NTILE(3) OVER (ORDER BY DATEDIFF(date('{max_dt}'), last_purchase_date) ASC) AS r_tile,  -- menor recency = mejor
    NTILE(3) OVER (ORDER BY frequency DESC) AS f_tile,
    NTILE(3) OVER (ORDER BY monetary DESC) AS m_tile
  FROM base
)
SELECT
  customer_id,
  first_purchase_date,
  last_purchase_date,
  recency_days,
  frequency,
  monetary,
  r_tile, f_tile, m_tile,
  CONCAT('R', r_tile, 'F', f_tile, 'M', m_tile) AS rfm_segment
FROM scored
""")
)

display(spark.table(f"{db}.gold_customer_rfm").orderBy("monetary DESC").limit(20))


### 3.3) gold_top_products_monthly — top 10 productos por país y mes

In [0]:

spark.sql(f"""
CREATE OR REPLACE TABLE {db}.gold_top_products_monthly AS
WITH base AS (
  SELECT
    date_trunc('month', sale_date) AS month_start,
    Country,
    Description,
    SUM(CASE WHEN is_return = false THEN line_total ELSE 0 END) AS revenue
  FROM {db}.retail_silver
  GROUP BY date_trunc('month', sale_date), Country, Description
),
ranked AS (
  SELECT
    month_start, Country, Description, revenue,
    ROW_NUMBER() OVER (PARTITION BY month_start, Country ORDER BY revenue DESC) AS rk
  FROM base
)
SELECT *
FROM ranked
WHERE rk <= 10
""")
)
display(spark.table(f"{db}.gold_top_products_monthly").orderBy("month_start","Country","rk"))


## 4) Vistas (views)
Crea al menos una vista persistente y una temporal.


In [0]:

# Vista persistente: clientes de alto valor
spark.sql(f"""
CREATE OR REPLACE VIEW {db}.v_rfm_high_value AS
SELECT *
FROM {db}.gold_customer_rfm
WHERE r_tile = 3 AND f_tile = 3 AND m_tile = 3
""")
)

# Vista temporal: ventas recientes (cambia la condición si deseas)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_sales_recent AS
SELECT *
FROM {db}.gold_daily_country
WHERE sale_date >= date_sub(current_date(), 7)
""")
display(spark.sql("SELECT * FROM v_sales_recent ORDER BY sale_date, Country"))


## 5) Operaciones Delta — History, Time Travel, OPTIMIZE/ZORDER, VACUUM

In [0]:

# History de tablas clave
for t in ["retail_bronze","retail_silver","gold_daily_country","gold_customer_rfm","gold_top_products_monthly"]:
    print("\n=== HISTORY:", t, "===")
    spark.sql(f"DESCRIBE HISTORY {db}.{t}").show(truncate=False)


In [0]:

# TODO (time travel): Ejecuta una actualización "errónea" controlada y luego consulta VERSION AS OF
# EJEMPLO (NO EJECUTES SIN ENTENDER): poner UnitPrice_dbl a NULL para un subconjunto pequeño y comparar.
# spark.sql(f"""
#   UPDATE {db}.retail_silver
#   SET UnitPrice_dbl = NULL
#   WHERE Country = 'United Kingdom' AND sale_date = date('2011-12-08') AND UnitPrice_dbl IS NOT NULL
# """)
#
# -- Ahora, inspecciona la versión anterior
# spark.sql(f"SELECT * FROM {db}.retail_silver VERSION AS OF 0 LIMIT 20").show()
#
# -- (Opcional) RESTORE TABLE
# spark.sql(f"RESTORE TABLE {db}.retail_silver TO VERSION AS OF <version_anterior>")


In [0]:

# OPTIMIZE + ZORDER (puede tardar según el entorno)
# TODO: Ejecuta si está disponible en tu edición
try:
    spark.sql(f"OPTIMIZE {db}.retail_silver ZORDER BY (CustomerID_int, sale_date)")
    spark.sql(f"OPTIMIZE {db}.gold_daily_country ZORDER BY (sale_date)")
except Exception as e:
    print("OPTIMIZE no disponible en tu edición o error:", e)


In [0]:

# VACUUM con retención segura (7 días). DRY RUN primero:
try:
    spark.sql(f"VACUUM {db}.retail_silver RETAIN 168 HOURS DRY RUN")
except Exception as e:
    print("VACUUM error:", e)


## 6) Validaciones y consultas finales

In [0]:

# TODO: Agrega aquí consultas que validen tus KPIs y muestren resultados clave
spark.sql(f"SELECT * FROM {db}.gold_daily_country ORDER BY sale_date DESC, Country LIMIT 20").show()
spark.sql(f"SELECT rfm_segment, COUNT(*) AS customers FROM {db}.gold_customer_rfm GROUP BY rfm_segment ORDER BY customers DESC").show()
spark.sql(f"SELECT * FROM {db}.gold_top_products_monthly ORDER BY month_start DESC, Country, rk LIMIT 20").show()


## 7) Conclusiones
**TODO:** Resume las decisiones de modelado (por qué esas reglas de calidad, por qué esas particiones/ZORDER, etc.) y hallazgos analíticos.
